# 08 · SMS ham — diagnostic annotation

**Цель:** диагностический аудит подвыборки SMS ham для понимания реальной структуры legitimate SMS корпуса.  
Результаты используются для проектирования prompt family `legitimate_sms` (§5.9, §6.7 `dataset_design_final.md`).

**Выход:** `data/interim/annotated/sms_ham_annotated.jsonl`

**Стратифицированная выборка:** 320 сообщений из 4 827 ham-строк, стратификация по `wc_bin`.

---

## Категории (`scenario_family`)

| Категория | Описание |
|---|---|
| `personal_everyday_sms` | Неформальная личная переписка — приветствия, личные социальные сообщения, личные планы |
| `coordination_or_logistics_sms` | Организационные сообщения — встречи, поездки, pick-up/drop-off, практическая координация |
| `service_notification_sms` | Доброкачественные автоматические уведомления от сервисов — OTP-коды, статус доставки, уведомления от известных банков/сервисов |
| `transactional_benign_sms` | Подтверждения покупок, квитанции о бронировании, сообщения о транзакциях |
| `mixed_or_unclear_sms` | Неоднозначно, охватывает несколько категорий или слишком короткое/повреждённое |

---

Аннотация через OpenRouter (`openai/gpt-4o-mini`). Resumable: уже аннотированные записи пропускаются через кэш по md5(text).


In [1]:
RANDOM_SEED = 42
N_SMS_HAM = 320
BATCH_SIZE = 12
SLEEP_SEC = 2.5
MAX_NEW_THIS_RUN = 320   # full sample in one pass; re-runs are no-ops (cache skips already annotated)
ANNOTATION_MODEL = "openai/gpt-4o-mini"



In [2]:
import importlib.util, json, os, time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, APIError, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm

def _find_v2_root() -> Path:
    c = Path(globals().get("__vsc_ipynb_file__", globals().get("__file__", "."))).resolve()
    for p in [c, *c.parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists(): return p
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists(): return p
    raise RuntimeError("v2 root")
V2_ROOT = _find_v2_root()
RAW_DIR = V2_ROOT / "data" / "raw" / "collected"
INTERIM = V2_ROOT / "data" / "interim" / "annotated"
BLOCK_AB = INTERIM / "block_ab"
SAMP_DIR = BLOCK_AB / "samples"
OUT_FLAT = INTERIM / "sms_ham_annotated.jsonl"
spec = importlib.util.spec_from_file_location("_ann_common", V2_ROOT / "notebooks" / "02_dataset_design" / "_ann_common.py")
ac = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ac)
load_dotenv(V2_ROOT / ".env" if (V2_ROOT / ".env").exists() else None)
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])



In [3]:
sms_all = ac.load_jsonl(RAW_DIR / "sms_spam.jsonl")
sms_ham = [r for r in sms_all if r.get("label") == "ham"]
def build():
    rows = [{**r, "_wc": ac.wc_bin(ac.wc(r.get("text") or ""))} for r in sms_ham]
    df = pd.DataFrame(rows)
    picked = ac.stratified_sample_df(df, N_SMS_HAM, ["_wc"], RANDOM_SEED + 17)
    return picked.drop(columns=["_wc"]).to_dict("records")
sample = ac.ensure_sample(SAMP_DIR / "sms_ham_stratified.jsonl", build)
print(len(sample))



320


In [4]:
SYSTEM_PROMPT = """\
You are annotating legitimate (non-spam) SMS messages for a research dataset on LLM-generated \
text detection in anti-fraud systems.

Classify each SMS into exactly one category based on its primary intent:

- personal_everyday_sms: casual personal conversation, greetings, social messages, personal plans
- coordination_or_logistics_sms: arranging meetings, travel, pick-up/drop-off, practical \
day-to-day coordination
- service_notification_sms: benign automated service notifications — OTP codes, delivery status, \
alerts from known banks/services the user trusts
- transactional_benign_sms: purchase confirmations, booking receipts, benign transaction messages
- mixed_or_unclear_sms: ambiguous, spans categories, or too short/garbled to classify reliably

Return strict JSON only — no explanation:
{"category": "<one of the five labels>", "confidence": "high|medium|low"}"""


def wrap(r):
    b = (r.get("text") or "").strip()
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": b},
    ]


OK = {"personal_everyday_sms", "coordination_or_logistics_sms", "service_notification_sms",
      "transactional_benign_sms", "mixed_or_unclear_sms"}


def val(d):
    c = d.get("category", "mixed_or_unclear_sms")
    if c not in OK:
        c = "mixed_or_unclear_sms"
    conf = d.get("confidence", "low")
    if conf not in {"high", "medium", "low"}:
        conf = "low"
    return {"category": c, "confidence": conf}


@retry(retry=retry_if_exception_type((RateLimitError, APIError)),
       wait=wait_exponential(multiplier=2, min=2, max=45), stop=stop_after_attempt(6))
def call_json(msgs):
    return json.loads(client.chat.completions.create(
        model=ANNOTATION_MODEL, messages=msgs,
        response_format={"type": "json_object"}, temperature=0, max_tokens=128,
    ).choices[0].message.content)



In [5]:
idx = ac.load_flat_annotation_index(OUT_FLAT)
pending = [r for r in sample if ac.md5_text_key(r.get("text","")) not in idx]
batch_i = 0
for r in tqdm(pending[:MAX_NEW_THIS_RUN], desc="sms_ham"):
    try:
        a = val(call_json(wrap(r)))
    except Exception as e:
        a = val({}); a["_error"] = str(e)[:200]
    ts = datetime.now(timezone.utc).isoformat()
    ex = {"_error": a["_error"]} if "_error" in a else None
    flat = ac.make_flat_record(r, scenario_family=a["category"], annotation_confidence=a["confidence"], annotation_model=ANNOTATION_MODEL, annotated_at=ts, extra=ex)
    ac.append_jsonl(OUT_FLAT, flat)
    batch_i += 1
    if batch_i >= BATCH_SIZE:
        batch_i = 0
        time.sleep(SLEEP_SEC)
print(len(ac.load_flat_annotation_index(OUT_FLAT)))



sms_ham:   0%|          | 0/320 [00:00<?, ?it/s]

318


In [6]:
# ── Diagnostic summary ──────────────────────────────────────────────────────
# Run this cell after the annotation loop to inspect the audit results.
# The category distribution and sample texts are the primary output used
# for designing the `legitimate_sms` prompt family (§6.7 dataset_design_final.md).

check = pd.DataFrame(ac.load_jsonl(OUT_FLAT))
if check.empty:
    print("No annotations yet — run the annotation loop first.")
else:
    print(f"Annotated: {len(check)} / target {N_SMS_HAM}")

    print("\n--- category distribution ---")
    dist = check["scenario_family"].value_counts()
    for cat, cnt in dist.items():
        print(f"  {cat:45s}  {cnt:4d}  ({cnt / len(check) * 100:.1f}%)")

    print("\n--- confidence distribution ---")
    print(check["annotation_confidence"].value_counts().to_string())

    errors = check.get("_error", pd.Series(dtype=str)).dropna()
    if len(errors):
        print(f"\n--- annotation errors: {len(errors)} ---")

    print("\n--- 2 samples per category ---")
    for cat in dist.index:
        print(f"\n== {cat} ==")
        subset = check.loc[check["scenario_family"] == cat, "text"].dropna()
        for txt in subset.head(2):
            print(f"  {str(txt)[:250]}")
        print()

Annotated: 320 / target 320

--- category distribution ---
  personal_everyday_sms                           189  (59.1%)
  mixed_or_unclear_sms                             96  (30.0%)
  coordination_or_logistics_sms                    35  (10.9%)

--- confidence distribution ---
annotation_confidence
high      210
medium     96
low        14

--- 2 samples per category ---

== personal_everyday_sms ==
  Even i cant close my eyes you are in me our vava playing umma :-D
  Watching cartoon, listening music &amp; at eve had to go temple &amp; church.. What about u?


== mixed_or_unclear_sms ==
  I hope your pee burns tonite.
  O was not into fps then.


== coordination_or_logistics_sms ==
  Ü dun need to pick ur gf?
  I'm at work. Please call

